# 03 — Model Öyrətmə və Submission

**Strategiya:**
- Baseline: TF-IDF (char n-gram) + Logistic Regression
- Güclü model: `paraphrase-multilingual-MiniLM-L12-v2` (118M param, CPU)
- Ansambl: soft voting (hər iki modelin ehtimallarının ortası)

> Yarış qaydası: inference yalnız CPU-da, maks 600M parametr, maks 8GB RAM

In [ ]:
import pandas as pd
import numpy as np
import random, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import sys
sys.path.append('../src')
from preprocess import clean_text, build_features

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

train['text'] = train.apply(build_features, axis=1)
test['text']  = test.apply(build_features, axis=1)

print('Train:', train.shape, '| Test:', test.shape)
print(train['label'].value_counts())

## 1. Baseline — TF-IDF + Logistic Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = train['text'].fillna('')
y = train['label']

tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        max_features=100_000,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
        random_state=SEED
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(tfidf_pipeline, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f'[TF-IDF Baseline] CV Macro F1: {scores.mean():.4f} ± {scores.std():.4f}')

tfidf_pipeline.fit(X, y)

## 2. Güclü Model — Multilingual Sentence Transformer

`paraphrase-multilingual-MiniLM-L12-v2`:
- 118M parametr (< 600M limit)
- AZ / RU / EN dəstəkləyir
- CPU-da inference edir
- Sentence-level embedding → LightGBM classifier

In [ ]:
from sentence_transformers import SentenceTransformer

# Model yüklənməsi (~420MB, ilk dəfə avtomatik endirilir)
embedder = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2',
    device='cpu'
)

print('Embedding yaradılır (train)...')
X_emb_train = embedder.encode(
    X.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print('Train embedding shape:', X_emb_train.shape)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

cv_emb = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores_emb = cross_val_score(lgbm, X_emb_train, y_enc, cv=cv_emb, scoring='f1_macro', n_jobs=-1)
print(f'[Transformer+LGBM] CV Macro F1: {scores_emb.mean():.4f} ± {scores_emb.std():.4f}')

lgbm.fit(X_emb_train, y_enc)

## 3. Ansambl — Soft Voting (TF-IDF + Transformer)

In [ ]:
X_test_text = test['text'].fillna('')

# TF-IDF proqnozları
prob_tfidf = tfidf_pipeline.predict_proba(X_test_text)
tfidf_classes = tfidf_pipeline.classes_  # ['customer_support','other','technical_support']

# Transformer embedding + LGBM proqnozları
print('Embedding yaradılır (test)...')
X_emb_test = embedder.encode(
    X_test_text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
prob_lgbm = lgbm.predict_proba(X_emb_test)
lgbm_classes = le.classes_  # ['customer_support','other','technical_support']

# Hər iki ehtimallıq matrisini uyğunlaşdır
assert list(tfidf_classes) == list(lgbm_classes), 'Class sırası uyğun deyil!'

# Soft voting: ağırlıqlı orta (transformer daha güclüdür)
w_tfidf = 0.35
w_lgbm  = 0.65
prob_ensemble = w_tfidf * prob_tfidf + w_lgbm * prob_lgbm

preds_idx = np.argmax(prob_ensemble, axis=1)
preds = tfidf_classes[preds_idx]

print('\nAnsambl label paylanması:')
print(pd.Series(preds).value_counts())

## 4. Submission Faylı Yarat

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'label': preds})

# Formatı yoxla
assert set(submission['label'].unique()).issubset({'technical_support','customer_support','other'}), 'Yanlış etiket!'
assert submission['id'].nunique() == len(submission), 'Dublikat ID var!'

os.makedirs('../outputs', exist_ok=True)
submission.to_csv('../outputs/submission.csv', index=False)
print('✅ Submission hazırdır: outputs/submission.csv')
submission.head(10)

## 5. Modeli Saxla (Tekrar İstifadə üçün)

In [ ]:
import pickle

with open('../outputs/tfidf_pipeline.pkl', 'wb') as f:
    pickle.dump(tfidf_pipeline, f)

with open('../outputs/lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm, f)

with open('../outputs/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('✅ Modellər saxlanıldı: outputs/')